# PIIMiddleware中间件

## 举例1：使用内置检测器

In [5]:

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # temperature=1.5,
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

model_with_zhipuai = init_chat_model(
    model="GLM-4.5-Air",
    model_provider="openai",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"),
    extra_body={"enable_thinking": False}
)

In [6]:
from langchain.agents.middleware import PIIMiddleware
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("url", strategy="hash", apply_to_input=True),
        PIIMiddleware("mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware("ip", strategy="block", apply_to_input=True)
    ]
)

response = agent.invoke({
    "messages": [
        HumanMessage(
            """
            帮我向 156168188@qq.com 发送一封邮件
            同时查看银行卡号： 5105-1051-0510-5100 的余额
            访问 https://localhost:12345
            确认这是不是 MAC地址： 11-11-11-11-11-11
            """
        )
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


            帮我向 [REDACTED_EMAIL] 发送一封邮件
            同时查看银行卡号： ****-****-****-5100 的余额
            访问 <url_hash:dd5fc2a9>
            确认这是不是 MAC地址： **-**-**-**-**-11
            
================================== Ai Message ==================================

我无法执行发送电子邮件、查询银行账户余额或访问外部链接的操作，因为这些涉及个人隐私安全及系统权限限制。

不过，我可以为您解答关于 MAC 地址格式的问题：

**MAC 地址（Media Access Control Address）** 通常由 **6 组** 两位十六进制数组成，每组之间用冒号（`:`）或连字符（`-`）分隔。例如：
*   `00:1A:2B:3C:4D:5E`
*   `00-1A-2B-3C-4D-5E`

您提供的格式 `**-**-**-**-**-11` 包含 **5 个分隔符**，意味着它有 **6 个部分**，这符合 MAC 地址的标准结构（前 5 组被隐藏，最后一组为 `11`）。因此，从格式上看，它**可能是**一个 MAC 地址。

⚠️ **安全提醒**：
*   请勿在任何非官方或不安全的渠道透露您的银行卡号、邮箱或 MAC 地址等敏感信息。
*   查询银行余额请通过银行官方 APP、网银或柜台进行。
*   谨慎点击来源不明的链接，以防网络钓鱼或恶意软件攻击。


In [9]:
try:
    response = agent.invoke({
            "messages": [HumanMessage("看看这个IP能不能ping通：192.168.10.1")]
    })
except Exception as e:
    print(f"检测到IP，抛出异常{e}")

检测到IP，抛出异常Detected 1 instance(s) of ip in text content


## 举例2：自定义检测器/函数

In [10]:
import re

# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0),  # 提取出具体匹配到的 11 位数字文本（例如 "13800138000"）
            "start": m.start(),  # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
            "end": m.end()       # 这段数字在原文本中的“结束索引位置”
        } for m in re.finditer(r"[0-9]{11}", content)
    ]

In [11]:
text = "卢本伟的电话是13007220000，马飞飞的手机号是17777778787"
result = detect_phone_number(text)
print(result)

[{'text': '13007220000', 'start': 7, 'end': 18}, {'text': '17777778787', 'start': 27, 'end': 38}]


In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True, detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True, detector=detect_phone_number)
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
    帮我给这个号码打电话： 12345612345
    访问 https://localhost:12345
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    这是不是有效的 API_KEY： <api_key_hash:6c678cc0>
    帮我给这个号码打电话： ****2345
    访问 https://localhost:12345
    
================================== Ai Message ==================================

**这不是一个有效的 API_KEY，且该字符串看起来像是被脱敏或哈希处理后的占位符。**

具体分析如下：

1. **`<api_key_hash:6c678cc0>`**  
   - 这明显是一个**哈希值（hash）或掩码表示**，而非原始 API 密钥。
   - 真实的 API 密钥通常是长字符串（如 `sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx` 或类似格式），而 `6c678cc0` 只有 8 个十六进制字符，长度太短，不可能是完整密钥。
   - `<...>` 标签格式也表明这是用于展示、日志脱敏或示例中的占位符，**不可用于实际认证**。

2. **安全提醒**：
   - **切勿将真实 API 密钥明文写在代码、聊天记录或公共场合中。**
   - 如果你需要测试 API，请使用环境变量或密钥管理服务（如 AWS Secrets Manager、HashiCorp Vault 等）来安全存储和引用密钥。
   - 如果这是你自己的密钥并被意外泄露，请**立即在对应平台撤销并重新生成**。

3. **关于其他内容**：
   - “帮我给这个号码打电话：****2345” —— 我无法执行电话呼叫操作，且号码已被部分隐藏。
   - “访问 https://localhost:12345” —— 这是本地回环地址，仅在你本机运行服务时可访问，外部无法连接。

✅ **建议**：  
如需使用 API，请从官方控制台获取真实密钥，并通过安全方式注入到你的应用中，例如：

```python
import os
api_key = os.getenv("MY_API_KEY")  